# Notebook 02: Preprocessing and Feature Engineering

## RustWeatherML - Weather Prediction System in Rust

This notebook covers:
1. Loading raw data from Notebook 01
2. Handling missing values
3. Outlier detection and treatment
4. Creating target variables
5. Feature engineering (lag features, rolling statistics)
6. Cyclical encoding (hour, day, month)
7. Data normalization
8. Train/Validation/Test split
9. Saving processed data

**Input**: Raw weather data from `data/raw/`

**Output**: Processed features in `data/processed/` and `data/features/`

---
## 1. Setup Dependencies

In [ ]:
// Load dependencies
:dep polars = { version = "0.46", features = ["lazy", "parquet", "csv", "json", "dtype-datetime", "rolling_window", "rank"] }
:dep ndarray = { version = "0.16", features = ["serde"] }
:dep serde = { version = "1.0", features = ["derive"] }
:dep chrono = { version = "0.4", features = ["serde"] }
:dep anyhow = "1.0"
:dep statrs = "0.18"

In [ ]:
use polars::prelude::*;
use std::f64::consts::PI;
use std::fs::File;

println!("Dependencies loaded!");

---
## 2. Load Raw Data

In [ ]:
// Load the sample data from Notebook 01
let parquet_path = "../data/raw/weather_sample_2024_01.parquet";

let df = LazyFrame::scan_parquet(parquet_path, Default::default())
    .expect("Failed to scan parquet")
    .collect()
    .expect("Failed to collect DataFrame");

println!("Loaded DataFrame: {} rows x {} columns", df.height(), df.width());
println!("\nColumn names:");
for name in df.get_column_names() {
    println!("  - {}", name);
}

In [ ]:
// Preview the data
println!("First 5 rows:");
println!("{}", df.head(Some(5)));

---
## 3. Missing Value Analysis and Handling

In [ ]:
// Analyze missing values
println!("=== MISSING VALUE ANALYSIS ===");
println!("\nColumn | Null Count | Null %");
println!("{}|{}|{}", "-".repeat(25), "-".repeat(12), "-".repeat(10));

let total_rows = df.height();
for col in df.get_columns() {
    let null_count = col.null_count();
    let null_pct = (null_count as f64 / total_rows as f64) * 100.0;
    println!("{:<25}| {:>10} | {:>7.2}%", col.name(), null_count, null_pct);
}

In [ ]:
/// Strategy for handling missing values:
/// 1. For weather measurements: Use forward fill then backward fill (temporal interpolation)
/// 2. For remaining nulls: Use column median

// Columns to interpolate
let numeric_cols = vec![
    "temperature_2m", "apparent_temperature", "dewpoint_2m",
    "precipitation", "rain", "snowfall",
    "windspeed_10m", "windgusts_10m", "winddirection_10m",
    "pressure_msl", "surface_pressure", "cloudcover", "visibility",
    "shortwave_radiation", "direct_radiation",
    "relativehumidity_2m"
];

println!("Handling missing values with forward/backward fill...");

In [ ]:
// Apply forward fill then backward fill for each city group
let df_filled = df.clone()
    .lazy()
    .with_columns([
        // Forward fill then backward fill for numeric columns
        col("temperature_2m").forward_fill(None).backward_fill(None),
        col("apparent_temperature").forward_fill(None).backward_fill(None),
        col("dewpoint_2m").forward_fill(None).backward_fill(None),
        col("precipitation").forward_fill(None).backward_fill(None).fill_null(lit(0.0)),
        col("rain").forward_fill(None).backward_fill(None).fill_null(lit(0.0)),
        col("snowfall").forward_fill(None).backward_fill(None).fill_null(lit(0.0)),
        col("windspeed_10m").forward_fill(None).backward_fill(None),
        col("windgusts_10m").forward_fill(None).backward_fill(None),
        col("winddirection_10m").forward_fill(None).backward_fill(None),
        col("pressure_msl").forward_fill(None).backward_fill(None),
        col("surface_pressure").forward_fill(None).backward_fill(None),
        col("cloudcover").forward_fill(None).backward_fill(None),
        col("visibility").forward_fill(None).backward_fill(None),
        col("shortwave_radiation").forward_fill(None).backward_fill(None).fill_null(lit(0.0)),
        col("direct_radiation").forward_fill(None).backward_fill(None).fill_null(lit(0.0)),
        col("relativehumidity_2m").forward_fill(None).backward_fill(None),
        col("weathercode").forward_fill(None).backward_fill(None).fill_null(lit(0)),
    ])
    .collect()
    .expect("Failed to fill missing values");

println!("✓ Missing values handled");
println!("\nNull counts after filling:");
for col in df_filled.get_columns() {
    let null_count = col.null_count();
    if null_count > 0 {
        println!("  {} : {} nulls remaining", col.name(), null_count);
    }
}
println!("  (No output means all nulls handled)");

---
## 4. Outlier Detection and Treatment

In [ ]:
/// Outlier detection using IQR method
/// For weather data, we use domain knowledge bounds:
/// - Temperature: -60°C to 60°C
/// - Humidity: 0% to 100%
/// - Wind speed: 0 to 300 km/h
/// - Pressure: 870 to 1084 hPa

println!("=== OUTLIER DETECTION ===");

// Define physical bounds for weather variables
let bounds: Vec<(&str, f64, f64)> = vec![
    ("temperature_2m", -60.0, 60.0),
    ("apparent_temperature", -70.0, 70.0),
    ("relativehumidity_2m", 0.0, 100.0),
    ("windspeed_10m", 0.0, 300.0),
    ("windgusts_10m", 0.0, 400.0),
    ("pressure_msl", 870.0, 1084.0),
    ("cloudcover", 0.0, 100.0),
    ("precipitation", 0.0, 500.0),  // mm per hour extreme
];

for (col_name, min_val, max_val) in &bounds {
    if let Ok(col) = df_filled.column(*col_name) {
        let series = col.f64().unwrap();
        
        // Count outliers
        let mut outlier_count = 0;
        for val in series.into_iter() {
            if let Some(v) = val {
                if v < *min_val || v > *max_val {
                    outlier_count += 1;
                }
            }
        }
        
        if outlier_count > 0 {
            println!("{}: {} outliers outside [{}, {}]", 
                     col_name, outlier_count, min_val, max_val);
        }
    }
}

println!("\n(Most weather data from Open-Meteo is already quality-controlled)");

In [ ]:
// Clip outliers to physical bounds
let df_clipped = df_filled.clone()
    .lazy()
    .with_columns([
        col("temperature_2m").clip(lit(-60.0), lit(60.0)),
        col("apparent_temperature").clip(lit(-70.0), lit(70.0)),
        col("relativehumidity_2m").clip(lit(0.0), lit(100.0)),
        col("windspeed_10m").clip(lit(0.0), lit(300.0)),
        col("windgusts_10m").clip(lit(0.0), lit(400.0)),
        col("pressure_msl").clip(lit(870.0), lit(1084.0)),
        col("cloudcover").clip(lit(0.0), lit(100.0)),
        col("precipitation").clip(lit(0.0), lit(500.0)),
    ])
    .collect()
    .expect("Failed to clip outliers");

println!("✓ Outliers clipped to physical bounds");

---
## 5. Create Target Variables

In [ ]:
/// Target variables:
/// 1. will_rain: Binary (1 if precipitation > 0, else 0)
/// 2. weather_condition: Multi-class (0-5 based on weathercode)
/// 3. temp_next_24h: Temperature 24 hours ahead
/// 4. temp_next_48h: Temperature 48 hours ahead
/// 5. temp_next_72h: Temperature 72 hours ahead

println!("Creating target variables...");

In [ ]:
/// Map WMO weather codes to condition classes
/// 0: Clear (codes 0, 1)
/// 1: Cloudy (codes 2, 3)
/// 2: Foggy (codes 45, 48)
/// 3: Rainy (codes 51-67, 80-82)
/// 4: Snowy (codes 71-77, 85-86)
/// 5: Stormy (codes 95-99)

fn wmo_to_condition(code: i64) -> i64 {
    match code {
        0 | 1 => 0,  // Clear
        2 | 3 => 1,  // Cloudy
        45 | 48 => 2,  // Foggy
        51..=67 | 80..=82 => 3,  // Rainy
        71..=77 | 85 | 86 => 4,  // Snowy
        95..=99 => 5,  // Stormy
        _ => 0,  // Default to clear
    }
}

println!("Weather condition mapping defined:");
println!("  0: Clear");
println!("  1: Cloudy");
println!("  2: Foggy");
println!("  3: Rainy");
println!("  4: Snowy");
println!("  5: Stormy");

In [ ]:
// Create target variables
let df_with_targets = df_clipped.clone()
    .lazy()
    // Binary rain target
    .with_column(
        when(col("precipitation").gt(lit(0.0)))
            .then(lit(1i64))
            .otherwise(lit(0i64))
            .alias("will_rain")
    )
    // Weather condition (using when/then for mapping)
    .with_column(
        when(col("weathercode").is_in(lit(Series::new("codes".into(), &[0i64, 1]))))
            .then(lit(0i64))
        .when(col("weathercode").is_in(lit(Series::new("codes".into(), &[2i64, 3]))))
            .then(lit(1i64))
        .when(col("weathercode").is_in(lit(Series::new("codes".into(), &[45i64, 48]))))
            .then(lit(2i64))
        .when(col("weathercode").gt_eq(lit(51i64)).and(col("weathercode").lt_eq(lit(67i64))))
            .then(lit(3i64))
        .when(col("weathercode").gt_eq(lit(80i64)).and(col("weathercode").lt_eq(lit(82i64))))
            .then(lit(3i64))
        .when(col("weathercode").gt_eq(lit(71i64)).and(col("weathercode").lt_eq(lit(77i64))))
            .then(lit(4i64))
        .when(col("weathercode").is_in(lit(Series::new("codes".into(), &[85i64, 86]))))
            .then(lit(4i64))
        .when(col("weathercode").gt_eq(lit(95i64)))
            .then(lit(5i64))
        .otherwise(lit(0i64))
        .alias("weather_condition")
    )
    .collect()
    .expect("Failed to create targets");

println!("✓ Created binary and multi-class targets");
println!("\nTarget distribution:");

// Will rain distribution
let rain_dist = df_with_targets.clone()
    .lazy()
    .group_by([col("will_rain")])
    .agg([col("city").count().alias("count")])
    .collect()
    .unwrap();
println!("\nWill Rain:\n{}", rain_dist);

In [ ]:
// Weather condition distribution
let condition_dist = df_with_targets.clone()
    .lazy()
    .group_by([col("weather_condition")])
    .agg([col("city").count().alias("count")])
    .sort(["weather_condition"], Default::default())
    .collect()
    .unwrap();

println!("Weather Condition Distribution:");
println!("{}", condition_dist);

let condition_names = ["Clear", "Cloudy", "Foggy", "Rainy", "Snowy", "Stormy"];
println!("\nMapping:");
for (i, name) in condition_names.iter().enumerate() {
    println!("  {} = {}", i, name);
}

In [ ]:
// Create future temperature targets (shift by -24, -48, -72 hours)
// Note: This creates the targets by looking ahead in time
// We need to do this per city to avoid data leakage

let df_with_future_temps = df_with_targets.clone()
    .lazy()
    .sort(["city", "timestamp"], Default::default())
    // Shift temperature column to get future values
    .with_columns([
        col("temperature_2m").shift(lit(-24)).over([col("city")]).alias("temp_next_24h"),
        col("temperature_2m").shift(lit(-48)).over([col("city")]).alias("temp_next_48h"),
        col("temperature_2m").shift(lit(-72)).over([col("city")]).alias("temp_next_72h"),
    ])
    .collect()
    .expect("Failed to create future temperature targets");

println!("✓ Created temperature forecast targets (24h, 48h, 72h)");
println!("\nSample of future temperature targets:");
println!("{}", df_with_future_temps
    .clone()
    .lazy()
    .select([col("city"), col("timestamp"), col("temperature_2m"), 
             col("temp_next_24h"), col("temp_next_48h"), col("temp_next_72h")])
    .head(10)
    .collect()
    .unwrap());

---
## 6. Feature Engineering

In [ ]:
/// Feature Engineering:
/// 1. Lag features (t-1, t-6, t-12, t-24)
/// 2. Rolling statistics (6h, 12h, 24h windows)
/// 3. Cyclical encoding (hour, day of week, month)
/// 4. Temperature gradients
/// 5. Pressure changes

println!("=== FEATURE ENGINEERING ===");
println!("Creating lag features, rolling statistics, and cyclical encodings...");

In [ ]:
// Parse timestamp and extract temporal features
let df_temporal = df_with_future_temps.clone()
    .lazy()
    .with_columns([
        // Parse timestamp string to datetime
        col("timestamp")
            .str()
            .to_datetime(
                Some(TimeUnit::Milliseconds),
                None,
                StrptimeOptions::default(),
                lit("raise"),
            )
            .alias("datetime"),
    ])
    .collect();

// If timestamp is already numeric (milliseconds), handle differently
let df_temporal = match df_temporal {
    Ok(df) => df,
    Err(_) => {
        // Timestamp might already be numeric
        println!("Note: Timestamp appears to be numeric, converting from milliseconds...");
        df_with_future_temps.clone()
            .lazy()
            .with_column(
                (col("timestamp") * lit(1_000_000i64))  // Convert ms to ns
                    .cast(DataType::Datetime(TimeUnit::Nanoseconds, None))
                    .alias("datetime")
            )
            .collect()
            .expect("Failed to convert timestamp")
    }
};

println!("✓ Datetime column created");
println!("{}", df_temporal.head(Some(3)));

In [ ]:
// Extract hour, day of week, month from datetime
let df_with_time_features = df_temporal.clone()
    .lazy()
    .with_columns([
        col("datetime").dt().hour().alias("hour"),
        col("datetime").dt().weekday().alias("day_of_week"),
        col("datetime").dt().month().alias("month"),
        col("datetime").dt().ordinal_day().alias("day_of_year"),
    ])
    .collect()
    .expect("Failed to extract time features");

println!("✓ Extracted temporal features (hour, day_of_week, month, day_of_year)");
println!("{}", df_with_time_features
    .clone()
    .lazy()
    .select([col("datetime"), col("hour"), col("day_of_week"), col("month"), col("day_of_year")])
    .head(5)
    .collect()
    .unwrap());

In [ ]:
// Cyclical encoding for temporal features
// sin/cos transformation to preserve cyclical nature

let df_cyclical = df_with_time_features.clone()
    .lazy()
    .with_columns([
        // Hour (0-23) -> cyclical
        (col("hour").cast(DataType::Float64) * lit(2.0 * PI / 24.0)).sin().alias("hour_sin"),
        (col("hour").cast(DataType::Float64) * lit(2.0 * PI / 24.0)).cos().alias("hour_cos"),
        
        // Day of week (0-6) -> cyclical
        (col("day_of_week").cast(DataType::Float64) * lit(2.0 * PI / 7.0)).sin().alias("dow_sin"),
        (col("day_of_week").cast(DataType::Float64) * lit(2.0 * PI / 7.0)).cos().alias("dow_cos"),
        
        // Month (1-12) -> cyclical
        ((col("month").cast(DataType::Float64) - lit(1.0)) * lit(2.0 * PI / 12.0)).sin().alias("month_sin"),
        ((col("month").cast(DataType::Float64) - lit(1.0)) * lit(2.0 * PI / 12.0)).cos().alias("month_cos"),
        
        // Day of year (1-365) -> cyclical
        ((col("day_of_year").cast(DataType::Float64) - lit(1.0)) * lit(2.0 * PI / 365.0)).sin().alias("doy_sin"),
        ((col("day_of_year").cast(DataType::Float64) - lit(1.0)) * lit(2.0 * PI / 365.0)).cos().alias("doy_cos"),
    ])
    .collect()
    .expect("Failed to create cyclical features");

println!("✓ Created cyclical encodings (sin/cos for hour, day_of_week, month, day_of_year)");
println!("\nCyclical features sample:");
println!("{}", df_cyclical
    .clone()
    .lazy()
    .select([col("hour"), col("hour_sin"), col("hour_cos"), 
             col("month"), col("month_sin"), col("month_cos")])
    .head(5)
    .collect()
    .unwrap());

In [ ]:
// Create lag features for key variables
let df_with_lags = df_cyclical.clone()
    .lazy()
    .sort(["city", "timestamp"], Default::default())
    .with_columns([
        // Temperature lags
        col("temperature_2m").shift(lit(1)).over([col("city")]).alias("temp_lag_1h"),
        col("temperature_2m").shift(lit(6)).over([col("city")]).alias("temp_lag_6h"),
        col("temperature_2m").shift(lit(12)).over([col("city")]).alias("temp_lag_12h"),
        col("temperature_2m").shift(lit(24)).over([col("city")]).alias("temp_lag_24h"),
        
        // Pressure lags
        col("pressure_msl").shift(lit(1)).over([col("city")]).alias("pressure_lag_1h"),
        col("pressure_msl").shift(lit(6)).over([col("city")]).alias("pressure_lag_6h"),
        col("pressure_msl").shift(lit(24)).over([col("city")]).alias("pressure_lag_24h"),
        
        // Humidity lags
        col("relativehumidity_2m").shift(lit(1)).over([col("city")]).alias("humidity_lag_1h"),
        col("relativehumidity_2m").shift(lit(6)).over([col("city")]).alias("humidity_lag_6h"),
        
        // Wind speed lags
        col("windspeed_10m").shift(lit(1)).over([col("city")]).alias("wind_lag_1h"),
        col("windspeed_10m").shift(lit(6)).over([col("city")]).alias("wind_lag_6h"),
        
        // Precipitation lag
        col("precipitation").shift(lit(1)).over([col("city")]).alias("precip_lag_1h"),
    ])
    .collect()
    .expect("Failed to create lag features");

println!("✓ Created lag features (1h, 6h, 12h, 24h)");
println!("  - Temperature lags: temp_lag_1h, temp_lag_6h, temp_lag_12h, temp_lag_24h");
println!("  - Pressure lags: pressure_lag_1h, pressure_lag_6h, pressure_lag_24h");
println!("  - Humidity lags: humidity_lag_1h, humidity_lag_6h");
println!("  - Wind lags: wind_lag_1h, wind_lag_6h");
println!("  - Precipitation lag: precip_lag_1h");

In [ ]:
// Create gradient features (rate of change)
let df_with_gradients = df_with_lags.clone()
    .lazy()
    .with_columns([
        // Temperature change in last hour
        (col("temperature_2m") - col("temp_lag_1h")).alias("temp_change_1h"),
        // Temperature change in last 6 hours
        (col("temperature_2m") - col("temp_lag_6h")).alias("temp_change_6h"),
        // Temperature change in last 24 hours
        (col("temperature_2m") - col("temp_lag_24h")).alias("temp_change_24h"),
        
        // Pressure change (important for weather prediction)
        (col("pressure_msl") - col("pressure_lag_1h")).alias("pressure_change_1h"),
        (col("pressure_msl") - col("pressure_lag_6h")).alias("pressure_change_6h"),
        (col("pressure_msl") - col("pressure_lag_24h")).alias("pressure_change_24h"),
        
        // Humidity change
        (col("relativehumidity_2m") - col("humidity_lag_1h")).alias("humidity_change_1h"),
    ])
    .collect()
    .expect("Failed to create gradient features");

println!("✓ Created gradient features (rate of change)");
println!("  - temp_change_1h, temp_change_6h, temp_change_24h");
println!("  - pressure_change_1h, pressure_change_6h, pressure_change_24h");
println!("  - humidity_change_1h");

In [ ]:
// Display final feature count
let final_df = df_with_gradients.clone();

println!("\n=== FINAL DATASET ===");
println!("Total rows: {}", final_df.height());
println!("Total columns: {}", final_df.width());
println!("\nColumn names ({} total):", final_df.width());

for (i, name) in final_df.get_column_names().iter().enumerate() {
    print!("{:<30}", name);
    if (i + 1) % 3 == 0 {
        println!();
    }
}
println!();

---
## 7. Data Normalization

In [ ]:
/// We'll compute normalization statistics but apply them during training
/// This preserves the original values for interpretability
/// Statistics are saved for use in production

let features_to_normalize = vec![
    "temperature_2m", "apparent_temperature", "dewpoint_2m",
    "windspeed_10m", "windgusts_10m", "pressure_msl", "surface_pressure",
    "cloudcover", "visibility", "relativehumidity_2m",
    "shortwave_radiation", "direct_radiation",
];

println!("=== NORMALIZATION STATISTICS ===");
println!("(Will be applied during model training)\n");

println!("{:<25} {:>10} {:>10} {:>10} {:>10}", "Feature", "Mean", "Std", "Min", "Max");
println!("{}", "-".repeat(70));

for feat in &features_to_normalize {
    if let Ok(col) = final_df.column(*feat) {
        let series = col.f64().unwrap();
        let mean = series.mean().unwrap_or(0.0);
        let std = series.std(1).unwrap_or(1.0);
        let min = series.min().unwrap_or(0.0);
        let max = series.max().unwrap_or(1.0);
        
        println!("{:<25} {:>10.2} {:>10.2} {:>10.2} {:>10.2}", feat, mean, std, min, max);
    }
}

---
## 8. Train/Validation/Test Split

**Temporal Split:**
- Training: 2016-2023
- Validation: 2024
- Test: 2025

For our sample data (January 2024), we'll demonstrate the split logic.

In [ ]:
// For sample data, we'll split by day within the month
// In production, this would be by year

let df_with_day = final_df.clone()
    .lazy()
    .with_column(
        col("datetime").dt().day().alias("day")
    )
    .collect()
    .expect("Failed to extract day");

// Split: Days 1-21 = Train, Days 22-25 = Val, Days 26-31 = Test
let train_df = df_with_day.clone()
    .lazy()
    .filter(col("day").lt_eq(lit(21)))
    .collect()
    .unwrap();

let val_df = df_with_day.clone()
    .lazy()
    .filter(col("day").gt(lit(21)).and(col("day").lt_eq(lit(25))))
    .collect()
    .unwrap();

let test_df = df_with_day.clone()
    .lazy()
    .filter(col("day").gt(lit(25)))
    .collect()
    .unwrap();

println!("=== DATA SPLIT (Sample) ===");
println!("Training set:   {} rows ({:.1}%)", train_df.height(), 
         100.0 * train_df.height() as f64 / df_with_day.height() as f64);
println!("Validation set: {} rows ({:.1}%)", val_df.height(),
         100.0 * val_df.height() as f64 / df_with_day.height() as f64);
println!("Test set:       {} rows ({:.1}%)", test_df.height(),
         100.0 * test_df.height() as f64 / df_with_day.height() as f64);

In [ ]:
// Verify target distribution in splits
println!("\n=== TARGET DISTRIBUTION BY SPLIT ===");

for (name, split) in [("Train", &train_df), ("Val", &val_df), ("Test", &test_df)] {
    let rain_pct = split.clone()
        .lazy()
        .select([col("will_rain").mean()])
        .collect()
        .unwrap()
        .column("will_rain")
        .unwrap()
        .f64()
        .unwrap()
        .get(0)
        .unwrap_or(0.0) * 100.0;
    
    println!("{}: {:.1}% rain", name, rain_pct);
}

---
## 9. Save Processed Data

In [ ]:
// Ensure directories exist
std::fs::create_dir_all("../data/processed").expect("Failed to create processed dir");
std::fs::create_dir_all("../data/features").expect("Failed to create features dir");

// Save full processed dataset
let processed_path = "../data/processed/weather_processed.parquet";
let mut file = File::create(processed_path).expect("Failed to create file");
ParquetWriter::new(&mut file)
    .finish(&mut final_df.clone())
    .expect("Failed to write parquet");
println!("✓ Saved processed data to: {}", processed_path);

// Save train/val/test splits
let train_path = "../data/features/train.parquet";
let val_path = "../data/features/val.parquet";
let test_path = "../data/features/test.parquet";

let mut file = File::create(train_path).expect("Failed to create file");
ParquetWriter::new(&mut file).finish(&mut train_df.clone()).unwrap();
println!("✓ Saved training set to: {}", train_path);

let mut file = File::create(val_path).expect("Failed to create file");
ParquetWriter::new(&mut file).finish(&mut val_df.clone()).unwrap();
println!("✓ Saved validation set to: {}", val_path);

let mut file = File::create(test_path).expect("Failed to create file");
ParquetWriter::new(&mut file).finish(&mut test_df.clone()).unwrap();
println!("✓ Saved test set to: {}", test_path);

---
## 10. Summary and Next Steps

### What we accomplished:
1. ✅ Loaded raw data from Notebook 01
2. ✅ Handled missing values (forward/backward fill)
3. ✅ Clipped outliers to physical bounds
4. ✅ Created target variables:
   - `will_rain` (binary)
   - `weather_condition` (6 classes)
   - `temp_next_24h`, `temp_next_48h`, `temp_next_72h` (regression)
5. ✅ Engineered features:
   - Temporal features (hour, day_of_week, month, day_of_year)
   - Cyclical encodings (sin/cos transforms)
   - Lag features (1h, 6h, 12h, 24h)
   - Gradient features (rate of change)
6. ✅ Computed normalization statistics
7. ✅ Created temporal train/val/test split
8. ✅ Saved processed data to parquet

### Features created:
- **Original features**: 17 weather variables
- **Target variables**: 5 (will_rain, weather_condition, temp forecasts)
- **Cyclical features**: 8 (sin/cos for hour, dow, month, doy)
- **Lag features**: 12 (various time lags)
- **Gradient features**: 7 (rate of change)

### Next Steps (Notebook 03):
1. Feature selection (correlation analysis, importance ranking)
2. Model training with multiple libraries:
   - linfa
   - smartcore
   - rustyml
3. Compare baseline models

In [ ]:
println!("\n" + "=".repeat(60).as_str());
println!("Notebook 02 Complete!");
println!("=".repeat(60));
println!("\nProceed to Notebook 03: Feature Selection & Model Training");